### 1. Imports et paramètres
On importe les bibliothèques et on fixe les paramètres qu'on pourra changer plus tard (fenêtre et fréquence de rééquilibrage).

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.covariance import LedoitWolf

# Paramètres
FICHIER = '../data/Data_SP.xlsx'
FENETRE = 252        # nombre de jours utilisés pour estimer la matrice (252 = 1 an)
FREQ = 'ME'          # fréquence de rééquilibrage : 'ME' = mensuel, 'QE' = trimestriel

### 2. Chargement des données
Dans le fichier Excel, chaque indice prend 3 colonnes (date, prix, colonne vide), et les données commencent à la 4e ligne. On récupère les prix de chaque indice et on les met dans un seul tableau, avec une colonne par indice.

In [19]:
brut = pd.read_excel(FICHIER, sheet_name='Données', header=None)

def convertir_date(x):
    # certaines dates sont stockées comme des nombres Excel
    if isinstance(x, (int, float)):
        return pd.Timestamp('1899-12-30') + pd.Timedelta(days=x)
    return pd.to_datetime(x, errors='coerce')

series = {}
for col in range(0, brut.shape[1], 3):
    nom = str(brut.iloc[0, col])
    if nom == 'nan':
        continue
    ticker = nom.replace(' Index', '')

    dates = brut.iloc[3:, col].apply(convertir_date)
    valeurs = pd.to_numeric(brut.iloc[3:, col + 1], errors='coerce')

    s = pd.Series(valeurs.values, index=dates.values).dropna()
    s = s[s.index.notna()]
    s = s[~s.index.duplicated()].sort_index()
    series[ticker] = s

prix = pd.DataFrame(series).sort_index()

print(prix.shape)
print(prix.index.min(), prix.index.max())

/usr/local/python/3.14.2/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


(8391, 26)
1992-01-02 00:00:00 2024-02-29 00:00:00


### 3. Rendements
On calcule les rendements journaliers des secteurs. Le S&P 500 (SPX) est mis à part : il sert seulement de référence pour comparer.

In [20]:
secteurs = [c for c in prix.columns if c != 'SPX']

rendements = prix[secteurs].pct_change(fill_method=None).iloc[1:]
rendements_spx = prix['SPX'].pct_change(fill_method=None).iloc[1:]

print(rendements.shape)
print(prix.notna().sum().sort_values().head(3))   # secteurs avec le moins de données

(8390, 25)
S5REDPIG     250
S5REAL      5843
S5PHRM      8391
dtype: int64


### 4. Matrices de covariance
On écrit une fonction par méthode : elle prend les rendements d'une fenêtre et renvoie la matrice de covariance.

La matrice de Ledoit-Wolf est un mélange entre la matrice empirique $\hat{\Sigma}$ et une matrice plus simple $F$ :
$$\hat{\Sigma}_{LW} = \delta F + (1-\delta)\hat{\Sigma}$$
Le coefficient $\delta$ est choisi automatiquement : plus la matrice empirique est bruitée, plus $\delta$ est grand.

In [21]:
def cov_empirique(r):
    return r.cov().values

def cov_ledoit_wolf(r):
    return LedoitWolf().fit(r.values).covariance_

# test sur la dernière fenêtre
test = rendements.tail(FENETRE).dropna(axis=1)   # on garde les secteurs sans donnée manquante
print(cov_empirique(test).shape, cov_ledoit_wolf(test).shape)
print('delta =', LedoitWolf().fit(test.values).shrinkage_)

(24, 24) (24, 24)
delta = 0.03584765683719352


### 5. Poids de variance minimale
On cherche les poids qui donnent le risque le plus faible, avec la somme des poids égale à 1 (les poids négatifs sont autorisés) :
$$w = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}^\top \Sigma^{-1}\mathbf{1}}$$

In [22]:
def poids_mv(cov):
    inv = np.linalg.pinv(cov)            # pseudo-inverse : marche même si la matrice est mal conditionnée
    uns = np.ones(cov.shape[0])
    return inv @ uns / (uns @ inv @ uns)

w_test = poids_mv(cov_ledoit_wolf(test))
print('somme des poids =', w_test.sum())

somme des poids = 0.9999999999999994


### 6. Calcul des poids dans le temps
À chaque fin de période (le dernier jour de Bourse du mois), on prend les `FENETRE` derniers jours, on calcule la matrice puis les poids.

Correction par rapport à la version 1 : on prend le dernier **vrai jour de Bourse** de chaque mois. Avant, quand la fin du mois tombait un week-end, le rééquilibrage était sauté.

In [23]:
# dernier jour de Bourse de chaque période
dates = pd.Series(rendements.index, index=rendements.index)
dates_reeq = dates.resample(FREQ).last()

def calcul_poids(methode_cov):
    liste = []
    for d in dates_reeq:
        r = rendements.loc[:d].tail(FENETRE).dropna(axis=1)
        if len(r) < 0.8 * FENETRE:      # pas assez de données au début
            continue
        w = poids_mv(methode_cov(r))
        liste.append(pd.Series(w, index=r.columns, name=d))
    return pd.DataFrame(liste).reindex(columns=secteurs).fillna(0)   # secteur absent = poids 0

poids_emp = calcul_poids(cov_empirique)
poids_lw = calcul_poids(cov_ledoit_wolf)

print(len(poids_emp), len(poids_lw))

377 377
